# Assignment 1

In this assignment, you'll be working with messy medical data and using regex to extract relevant infromation from the data. 

Each line of the `dates.txt` file corresponds to a medical note. Each note has a date that needs to be extracted, but each date is encoded in one of many formats.

The goal of this assignment is to correctly identify all of the different date variants encoded in this dataset and to properly normalize and sort the dates. 

Here is a list of some of the variants you might encounter in this dataset:
* 04/20/2009; 04/20/09; 4/20/09; 4/3/09
* Mar-20-2009; Mar 20, 2009; March 20, 2009;  Mar. 20, 2009; Mar 20 2009;
* 20 Mar 2009; 20 March 2009; 20 Mar. 2009; 20 March, 2009
* Mar 20th, 2009; Mar 21st, 2009; Mar 22nd, 2009
* Feb 2009; Sep 2009; Oct 2010
* 6/2008; 12/2009
* 2009; 2010

Once you have extracted these date patterns from the text, the next step is to sort them in ascending chronological order accoring to the following rules:
* Assume all dates in xx/xx/xx format are mm/dd/yy
* Assume all dates where year is encoded in only two digits are years from the 1900's (e.g. 1/5/89 is January 5th, 1989)
* If the day is missing (e.g. 9/2009), assume it is the first day of the month (e.g. September 1, 2009).
* If the month is missing (e.g. 2010), assume it is the first of January of that year (e.g. January 1, 2010).
* Watch out for potential typos as this is a raw, real-life derived dataset.

With these rules in mind, find the correct date in each note and return a pandas Series in chronological order of the original Series' indices. **This Series should be sorted by a tie-break sort in the format of ("extracted date", "original row number").**

For example if the original series was this:

    0    1999
    1    2010
    2    1978
    3    2015
    4    1985

Your function should return this:

    0    2
    1    4
    2    0
    3    1
    4    3

Your score will be calculated using [Kendall's tau](https://en.wikipedia.org/wiki/Kendall_rank_correlation_coefficient), a correlation measure for ordinal data.

*This function should return a Series of length 500 and dtype int.*

In [1]:
import pandas as pd

doc = []
with open('assets/dates.txt') as file:
    for line in file:
        doc.append(line)

df = pd.Series(doc)
df.head(10)

0         03/25/93 Total time of visit (in minutes):\n
1                       6/18/85 Primary Care Doctor:\n
2    sshe plans to move as of 7/8/71 In-Home Servic...
3                7 on 9/27/75 Audit C Score Current:\n
4    2/6/96 sleep studyPain Treatment Pain Level (N...
5                    .Per 7/06/79 Movement D/O note:\n
6    4, 5/18/78 Patient's thoughts about current su...
7    10/24/89 CPT Code: 90801 - Psychiatric Diagnos...
8                         3/7/86 SOS-10 Total Score:\n
9             (4/10/71)Score-1Audit C Score Current:\n
dtype: str

In [43]:
def date_sorter():
    from datetime import datetime
    import re

    month_alias = {
        'jan': 1, 'january': 1,
        'feb': 2, 'february': 2,
        'mar': 3, 'march': 3,
        'apr': 4, 'april': 4,
        'may': 5,
        'jun': 6, 'june': 6,
        'jul': 7, 'july': 7,
        'aug': 8, 'august': 8,
        'sep': 9, 'sept': 9, 'september': 9,
        'oct': 10, 'october': 10,
        'nov': 11, 'november': 11,
        'dec': 12, 'december': 12,
    }
    typo_fix = {
        'janaury': 'january',
        'decemeber': 'december',
    }

    def month_num(token: str):
        t = token.lower().strip('.,;:-()[]{}')
        t = typo_fix.get(t, t)
        if t in month_alias:
            return month_alias[t]
        if len(t) > 3 and t[1:] in month_alias:
            return month_alias[t[1:]]
        return None

    parsed_dates = []

    for i, text in enumerate(df):
        date = None

        # 1) mm/dd/yyyy or mm/dd/yy
        m = re.search(r'(?<!\d)0*(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})(?!\d)', text)
        if m:
            mo, day, yr = int(m.group(1)), int(m.group(2)), int(m.group(3))
            if yr < 100:
                yr += 1900
            try:
                date = datetime(yr, mo, day)
            except ValueError:
                pass

        # 2) Month dd, yyyy / Mon. dd yyyy / Mar 20th, 2009
        if date is None:
            m = re.search(
                r'(?<!\d)([A-Za-z]{3,12})\.?\s*-?\s*(\d{1,2})(?:st|nd|rd|th)?(?:,|\s)+(\d{2,4})(?!\d)',
                text,
                flags=re.IGNORECASE,
            )
            if m:
                mo = month_num(m.group(1))
                day, yr = int(m.group(2)), int(m.group(3))
                if yr < 100:
                    yr += 1900
                if mo is not None:
                    try:
                        date = datetime(yr, mo, day)
                    except ValueError:
                        pass

        # 3) dd Month yyyy / 20 March, 2009
        if date is None:
            m = re.search(
                r'(?<!\d)(\d{1,2})\s*-?\s*([A-Za-z]{3,12})\.?\s*(?:,|\s)+(\d{2,4})(?!\d)',
                text,
                flags=re.IGNORECASE,
            )
            if m:
                day = int(m.group(1))
                mo = month_num(m.group(2))
                yr = int(m.group(3))
                if yr < 100:
                    yr += 1900
                if mo is not None:
                    try:
                        date = datetime(yr, mo, day)
                    except ValueError:
                        pass

        # 4) Month yyyy (no day -> day=1)
        if date is None:
            m = re.search(r'(?<!\d)([A-Za-z]{3,12})\.?\s*(?:,|\s)+(\d{4})(?!\d)', text, flags=re.IGNORECASE)
            if m:
                mo = month_num(m.group(1))
                yr = int(m.group(2))
                if mo is not None:
                    date = datetime(yr, mo, 1)

        # 5) m/yyyy (no day -> day=1)
        if date is None:
            m = re.search(r'(?<!\d)(\d{1,2})/(\d{4})(?!\d)', text)
            if m:
                mo, yr = int(m.group(1)), int(m.group(2))
                if 1 <= mo <= 12:
                    date = datetime(yr, mo, 1)

        # 6) yyyy only (month/day missing -> Jan 1)
        if date is None:
            m = re.search(r'(?<!\d)((?:19|20)\d{2})(?!\d)', text)
            if m:
                yr = int(m.group(1))
                date = datetime(yr, 1, 1)

        parsed_dates.append((date, i))

    parsed_dates.sort(key=lambda x: (x[0] if x[0] is not None else datetime(9999, 1, 1), x[1]))
    return pd.Series([idx for _, idx in parsed_dates], dtype=int)

In [44]:
date_sorter()

0        9
1       84
2        2
3       53
4       28
      ... 
495    427
496    141
497    186
498    161
499    413
Length: 500, dtype: int64

In [45]:
def mult_sol(func):
    """ tries multiple solutions """
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

In [46]:
import re
import numpy as np
def setup(sol: pd.Series):
    s_test = sol()
    return s_test

def run_df_modified_check():
    """
    Check if df appears to be modified.
    """
    try:
        assert type(df) == pd.Series
        assert (df.index == pd.RangeIndex(start=0, stop=500, step=1)).all()
        assert (df.apply(type) == str).all()
        assert df.str.len().min() >= 6
        assert df.str[5].apply(ord).sum() == 38354
        print("Passed df modification check")
    except:
        print("Failed df modification check")


# check if running the code twice produces the same result
def check_repeatability(func, test_func):
    """
    Check if running the code twice produces the same result.
    The results of the test are printed.
    Args:
    func: function to run
    Returns:
    None
    """
    try:
        assert (func() == test_func).all()
        print("Passed repeatability check")
    except:
        print("Failed repeatability check")

# check if the result has the expected index
def check_index(func):
    """Check if the result has the expected index."""
    try:
        # assert type(date_sorter().index) == pd.RangeIndex
        # assert (date_sorter().index == pd.RangeIndex(start=0, stop=500, step=1)).all()
        assert list(func().index) == list(range(500))
        print("Passed index check")
    except:
        print("Failed index check")

# check the tie-break sort for a sample of records where some have the same date
# note that this only tests a sample and does not check the entire answer
def check_secondary_sort(s_test):
    try:
        test_indices = [335, 415, 323, 405, 370, 382, 303, 488, 283,
                    395, 318, 369, 493, 252, 314, 410, 490]
        answer_lkp = {original_index:answer_index for
                  answer_index, original_index in s_test().to_dict().items()}
        i_test = [answer_lkp[i] for i in test_indices]
        assert sorted(i_test) == i_test
        print("Passed secondary sort sample check")
    except:
        print("Failed secondary sort sample check")



def run_v_check(func):
    """
    Check if the parsed dates appear to be correct and correctly sorted.
    The check works by producing some test checksums
    if you get for example a False entry in the agree column for
    index value 20 that would mean you have at least one incorrectly
    parsed or incorrectly sorted date in the **output** index
    range 20,21,...,29
    The results of the test are printed.
    Args:
    s_test: Series such as produced by date_sorter()
    Returns:
    None
    """
    def wrapper(*args, **kwargs):
        try:
            func(*args, **kwargs)
        except:
            print("Failed values check")
    return wrapper

def test(sol: pd.Series):
    """Runs values check sums"""
    v_check = pd.DataFrame({'correct':
            [6695, 14428, 16742, 9275, 12290, 14654, 9421, 10185, 11464, 16491,
            11797, 14036, 15459, 9412, 13069, 10400, 10498, 14322, 13274, 11001,
            11383, 11910, 10977, 9692, 10199, 10187, 15456, 13491, 9186, 13646,
            11142, 13724, 10994, 12905, 15968, 16648, 13966, 14607, 16932, 14622,
            17942, 18220, 17818, 18305, 19633, 12522, 13978, 18445, 20156, 14797],
            'learner':[
            (sol.iloc[10*i:(i+1)*10].values * np.array(range(1,11))).sum() for i in range(50)]},
        index=range(0,500,10)).assign(agree=lambda x:x['correct']==x['learner'])
    print("Values checksums:")
    print(v_check["agree"].value_counts())
    assert v_check['agree'].all()
    print("Passed values check")

In [41]:
def check_secondary_sort(s_test):
    try:
        test_indices = [335, 415, 323, 405, 370, 382, 303, 488, 283,
                        395, 318, 369, 493, 252, 314, 410, 490]

        result = s_test() if callable(s_test) else s_test
        answer_lkp = {original_index: answer_index for answer_index, original_index in result.to_dict().items()}
        i_test = [answer_lkp[i] for i in test_indices]

        assert sorted(i_test) == i_test
        print("Passed secondary sort sample check")
    except:
        print("Failed secondary sort sample check")

In [47]:
print("Running checks...")

run_df_modified_check()
check_repeatability(date_sorter, setup(date_sorter))
check_index(date_sorter)
check_secondary_sort(setup(date_sorter))
run_v_check(test)(setup(date_sorter))

Running checks...
Passed df modification check
Passed repeatability check
Passed index check
Failed secondary sort sample check
Values checksums:
agree
True    50
Name: count, dtype: int64
Passed values check


In [ ]:
from scipy.stats import kendalltau

def date_sorter_kendalltau(x: pd.Series, y: pd.Series):
    tau, p_value = kendalltau(x, y)
    return tau, p_value

In [48]:
sample_idx = [335, 415, 323, 405, 370, 382, 303, 488, 283, 395, 318, 369, 493, 252, 314, 410, 490]

import pandas as pd
pd.DataFrame({"idx": sample_idx, "text": df.iloc[sample_idx].values})

,idx,text
0,335,"Open appendectomy, as a child. 2. Laparoscopi..."
1,415,2/1973 Primary Care Doctor:\n
2,323,4ectopic pregnancy in March 1973 Pertinent Med...
3,405,"History of two suicide attempts, most recently..."
4,370,4 (9/1975)Patient's thoughts about current sub...
5,382,sOne prior voluntary hospitalization in 09/197...
6,303,"7 first psych hospital approx age 20, most rec..."
7,488,tProblems renal cell cancer : s/p nephrectomy ...
8,283,AFeb 1977: Symmes Hospital\n
9,395,2/1977 Primary Care Doctor:\n


In [31]:
v_check = pd.DataFrame({
    'correct': [6695, 14428, 16742, 9275, 12290, 14654, 9421, 10185, 11464, 16491,
                11797, 14036, 15459, 9412, 13069, 10400, 10498, 14322, 13274, 11001,
                11383, 11910, 10977, 9692, 10199, 10187, 15456, 13491, 9186, 13646,
                11142, 13724, 10994, 12905, 15968, 16648, 13966, 14607, 16932, 14622,
                17942, 18220, 17818, 18305, 19633, 12522, 13978, 18445, 20156, 14797],
    'learner': [(date_sorter().iloc[10*i:(i+1)*10].values * np.array(range(1, 11))).sum() for i in range(50)]
}, index=range(0, 500, 10)).assign(delta=lambda x: x['learner'] - x['correct'])

v_check[v_check['delta'] != 0]

,correct,learner,delta
80,11464,12104,640
90,16491,15870,-621


In [34]:
# inspect records around the mismatch window
from datetime import datetime
import re

s = date_sorter()
window_idx = s.iloc[75:106].tolist()

MONTH_RE = (
    r'(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|'
    r'Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|'
    r'Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\.?'
)

def month_num(token: str) -> int:
    t = token.lower().rstrip('.')
    if t.startswith('jan'): return 1
    if t.startswith('feb'): return 2
    if t.startswith('mar'): return 3
    if t.startswith('apr'): return 4
    if t.startswith('may'): return 5
    if t.startswith('jun'): return 6
    if t.startswith('jul'): return 7
    if t.startswith('aug'): return 8
    if t.startswith('sep'): return 9
    if t.startswith('oct'): return 10
    if t.startswith('nov'): return 11
    return 12

def parse_one(text):
    date = None
    m = re.search(r'(?<!\d)0*(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})(?!\d)', text)
    if m:
        mo, day, yr = int(m.group(1)), int(m.group(2)), int(m.group(3))
        if yr < 100:
            yr += 1900
        try:
            date = datetime(yr, mo, day)
        except ValueError:
            pass
    if date is None:
        m = re.search(rf'({MONTH_RE})\s*-?\s*(\d{{1,2}})(?:st|nd|rd|th)?(?:,|\s)+(\d{{2,4}})', text, re.IGNORECASE)
        if m:
            mo = month_num(m.group(1)); day = int(m.group(2)); yr = int(m.group(3));
            if yr < 100: yr += 1900
            try: date = datetime(yr, mo, day)
            except ValueError: pass
    if date is None:
        m = re.search(rf'(?<!\d)(\d{{1,2}})\s*-?\s*({MONTH_RE})(?:,|\s)+(\d{{2,4}})', text, re.IGNORECASE)
        if m:
            day = int(m.group(1)); mo = month_num(m.group(2)); yr = int(m.group(3));
            if yr < 100: yr += 1900
            try: date = datetime(yr, mo, day)
            except ValueError: pass
    if date is None:
        m = re.search(rf'({MONTH_RE})(?:,|\s)+(\d{{4}})', text, re.IGNORECASE)
        if m:
            date = datetime(int(m.group(2)), month_num(m.group(1)), 1)
    if date is None:
        m = re.search(r'(?<!\d)(\d{1,2})/(\d{4})(?!\d)', text)
        if m:
            mo, yr = int(m.group(1)), int(m.group(2))
            if 1 <= mo <= 12: date = datetime(yr, mo, 1)
    if date is None:
        m = re.search(r'(?<!\d)((?:19|20)\d{2})(?!\d)', text)
        if m: date = datetime(int(m.group(1)), 1, 1)
    return date

pd.DataFrame({
    'rank': list(range(75,106)),
    'idx': window_idx,
    'date': [parse_one(df.iloc[i]) for i in window_idx],
    'text': [df.iloc[i] for i in window_idx]
})[['rank','idx','date','text']]


,rank,idx,date,text
0,75,123,1977-05-04,5/04/77 CPT Code: 90792: With medical services\n
1,76,19,1977-05-21,) 59 yo unemployed w referred by Urgent Care f...
2,77,117,1977-06-20,4 (6/20/77)Audit C Score Current:\n
3,78,232,1977-07-01,")HTN, hypercholesterolemia, DM, sleep apnea,, ..."
4,79,72,1977-07-11,Lithium 0.25 (7/11/77). LFTS wnl. Urine tox ...
5,80,189,1977-10-21,ssee 21 Oct 1977 Schroder Hospital discharge s...
6,81,313,1978-01-01,nLoss of father to cardiac event in Decemeber ...
7,82,318,1978-01-01,) Paxil (Jan 1978) : sedation\n
8,83,369,1978-01-01,1/1978 Primary Care Doctor:\n
9,84,493,1978-01-01,r1978\n


In [49]:
# Find month-like tokens before 4-digit years that are not in standard month names
std = {'jan','january','feb','february','mar','march','apr','april','may','jun','june','jul','july','aug','august','sep','sept','september','oct','october','nov','november','dec','december'}

cands = []
for i, t in enumerate(df):
    for m in re.finditer(r'([A-Za-z]{3,15})\s+(19\d{2}|20\d{2})', t):
        token = m.group(1).lower().rstrip('.')
        if token not in std:
            cands.append((i, token, m.group(2), t.strip()))

pd.DataFrame(cands, columns=['idx','token','year','text']).head(40)

,idx,token,year,text
0,235,poct,2015,pOct 2015 - Admitted to Gray Clinic for depres...
1,245,lnovember,1990,lNovember 1990 - NPCCHx of Outpatient Treatmen...
2,256,iaug,1988,IAug 1988: Symmes Hospital s/p SA OD with ASA
3,281,yaug,2004,yAug 2004 - Dr. Tom Ngo at TMC; dx GAD; patien...
4,283,afeb,1977,AFeb 1977: Symmes Hospital
5,285,ssep,1983,"sSep 1983 GSW to face (L-TMJ region), ? gang r..."
6,298,janaury,1993,"HH, Janaury 1993"
7,313,decemeber,1978,nLoss of father to cardiac event in Decemeber ...
8,333,snovember,1997,sNovember 1997 - suicidal ideation - HHR
9,459,ssince,1998,sSince 1998. Prior medication trials (includin...


In [ ]:
@mult_sol
def date_sorter():
    
    order = None
    # YOUR CODE HERE
    from datetime import datetime
    import re

    month_alias = {
        'jan': 1, 'january': 1,
        'feb': 2, 'february': 2,
        'mar': 3, 'march': 3,
        'apr': 4, 'april': 4,
        'may': 5,
        'jun': 6, 'june': 6,
        'jul': 7, 'july': 7,
        'aug': 8, 'august': 8,
        'sep': 9, 'sept': 9, 'september': 9,
        'oct': 10, 'october': 10,
        'nov': 11, 'november': 11,
        'dec': 12, 'december': 12,
    }
    typo_fix = {
        'janaury': 'january',
        'decemeber': 'december',
    }

    def month_num(token: str):
        t = token.lower().strip('.,;:-()[]{}')
        t = typo_fix.get(t, t)
        if t in month_alias:
            return month_alias[t]
        if len(t) > 3 and t[1:] in month_alias:
            return month_alias[t[1:]]
        return None

    parsed_dates = []

    for i, text in enumerate(df):
        date = None

        # 1) mm/dd/yyyy or mm/dd/yy
        m = re.search(r'(?<!\d)0*(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})(?!\d)', text)
        if m:
            mo, day, yr = int(m.group(1)), int(m.group(2)), int(m.group(3))
            if yr < 100:
                yr += 1900
            try:
                date = datetime(yr, mo, day)
            except ValueError:
                pass

        # 2) Month dd, yyyy / Mon. dd yyyy / Mar 20th, 2009
        if date is None:
            m = re.search(
                r'(?<!\d)([A-Za-z]{3,12})\.?\s*-?\s*(\d{1,2})(?:st|nd|rd|th)?(?:,|\s)+(\d{2,4})(?!\d)',
                text,
                flags=re.IGNORECASE,
            )
            if m:
                mo = month_num(m.group(1))
                day, yr = int(m.group(2)), int(m.group(3))
                if yr < 100:
                    yr += 1900
                if mo is not None:
                    try:
                        date = datetime(yr, mo, day)
                    except ValueError:
                        pass

        # 3) dd Month yyyy / 20 March, 2009
        if date is None:
            m = re.search(
                r'(?<!\d)(\d{1,2})\s*-?\s*([A-Za-z]{3,12})\.?\s*(?:,|\s)+(\d{2,4})(?!\d)',
                text,
                flags=re.IGNORECASE,
            )
            if m:
                day = int(m.group(1))
                mo = month_num(m.group(2))
                yr = int(m.group(3))
                if yr < 100:
                    yr += 1900
                if mo is not None:
                    try:
                        date = datetime(yr, mo, day)
                    except ValueError:
                        pass

        # 4) Month yyyy (no day -> day=1)
        if date is None:
            m = re.search(r'(?<!\d)([A-Za-z]{3,12})\.?\s*(?:,|\s)+(\d{4})(?!\d)', text, flags=re.IGNORECASE)
            if m:
                mo = month_num(m.group(1))
                yr = int(m.group(2))
                if mo is not None:
                    date = datetime(yr, mo, 1)

        # 5) m/yyyy (no day -> day=1)
        if date is None:
            m = re.search(r'(?<!\d)(\d{1,2})/(\d{4})(?!\d)', text)
            if m:
                mo, yr = int(m.group(1)), int(m.group(2))
                if 1 <= mo <= 12:
                    date = datetime(yr, mo, 1)

        # 6) yyyy only (month/day missing -> Jan 1)
        if date is None:
            m = re.search(r'(?<!\d)((?:19|20)\d{2})(?!\d)', text)
            if m:
                yr = int(m.group(1))
                date = datetime(yr, 1, 1)

        parsed_dates.append((date, i))

    parsed_dates.sort(key=lambda x: (x[0] if x[0] is not None else datetime(9999, 1, 1), x[1]))
    order = pd.Series([idx for _, idx in parsed_dates], dtype=int)
    
    return order # Your answer here